In [1]:
# Setup: load outputs, check benchmark vs mandate, inspect latest views
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "data.py").exists())
sys.path.insert(0, str(ROOT))
from src import data

TICKERS = data.TICKERS
EQUITY, BONDS, ALTS = ["SPY", "EFA", "EEM"], ["IEF", "LQD", "TLT"], ["VNQ", "GLD"]
BENCH = pd.Series({"SPY": 0.30, "EFA": 0.12, "EEM": 0.08, "IEF": 0.15,
                   "LQD": 0.10, "TLT": 0.10, "VNQ": 0.08, "GLD": 0.07})[TICKERS]

clean, daily, monthly = data.load_processed()
views = pd.read_csv(data.PROCESSED_DIR / "bl_views.csv")
views["Month"] = pd.PeriodIndex(views["Month"], freq="M")

# --- Data checks
print("Daily returns:", daily.index.min().date(), "->", daily.index.max().date(),
      "| rows:", len(daily), "| irregular:", int((~daily["regular"]).sum()))
print("Monthly returns:", monthly.index.min(), "->", monthly.index.max(), "| rows:", len(monthly))
print("View months:", views["Month"].min(), "->", views["Month"].max(),
      "| views per month:", views.groupby("Month").size().unique())

# --- The benchmark must satisfy the mandate, otherwise 'no views -> benchmark' is impossible
checks = {
    "sum = 1": bool(np.isclose(BENCH.sum(), 1.0)),
    "long-only": bool((BENCH >= 0).all()),
    "max weight <= 40%": bool(BENCH.max() <= 0.40),
    "bonds in [25%, 45%]": bool(0.25 <= BENCH[BONDS].sum() <= 0.45),
    "EEM <= 1/3 equity": bool(BENCH["EEM"] <= BENCH[EQUITY].sum() / 3),
}
print("\nBenchmark vs mandate:", checks)
print(f"Bonds {BENCH[BONDS].sum():.0%} | Equity {BENCH[EQUITY].sum():.0%} | "
      f"EEM/Equity {BENCH['EEM'] / BENCH[EQUITY].sum():.1%}")

# --- Latest views (P rows, Q = annual expected return spread, decimal)
print("\nViews for", views["Month"].max())
print(views.loc[views["Month"] == views["Month"].max(), ["View", *TICKERS, "Q"]]
      .round(4).to_string(index=False))

Daily returns: 2007-01-04 -> 2026-11-04 | rows: 4991 | irregular: 0
Monthly returns: 2007-02 -> 2026-10 | rows: 237
View months: 2008-01 -> 2026-10 | views per month: [4]

Benchmark vs mandate: {'sum = 1': True, 'long-only': True, 'max weight <= 40%': True, 'bonds in [25%, 45%]': True, 'EEM <= 1/3 equity': True}
Bonds 35% | Equity 50% | EEM/Equity 16.0%

Views for 2026-10
                            View  SPY  EFA  EEM     IEF     LQD     TLT  VNQ  GLD      Q
    Momentum Equities: EEM > SPY -1.0 0.00 1.00  0.0000  0.0000  0.0000  0.0  0.0 0.0162
       Momentum Bonds: LQD > TLT  0.0 0.00 0.00  0.0000  1.0000 -1.0000  0.0  0.0 0.0097
Momentum Alternatives: VNQ > GLD  0.0 0.00 0.00  0.0000  0.0000  0.0000  1.0 -1.0 0.0014
        Trend: equities vs bonds  0.6 0.24 0.16 -0.4286 -0.2857 -0.2857  0.0  0.0 0.0144


In [2]:
# Covariance: Ledoit-Wolf (2003) shrinkage to constant correlation, trailing 252 regular daily returns
def ledoit_wolf_cc(X):
    """Ledoit-Wolf 'Honey, I shrunk...' estimator: Sigma = delta*F + (1-delta)*S, F = constant-correlation target.
    X: (T, N) daily returns. Returns (Sigma, delta, rbar) in the units of X (daily)."""
    X = np.asarray(X, float)
    T, N = X.shape
    X = X - X.mean(axis=0)
    S = X.T @ X / T                                   # sample covariance (MLE, as in LW)
    var = np.diag(S); sd = np.sqrt(var)
    rbar = ((S / np.outer(sd, sd)).sum() - N) / (N * (N - 1))   # average off-diagonal correlation
    F = rbar * np.outer(sd, sd); np.fill_diagonal(F, var)

    # Optimal shrinkage intensity kappa = (pi - rho) / gamma, delta = clip(kappa / T, 0, 1)
    X2 = X ** 2
    pi_mat = X2.T @ X2 / T - S ** 2                   # asy var of each s_ij
    theta = (X ** 3).T @ X / T - var[:, None] * S     # asy cov(s_ii, s_ij)
    np.fill_diagonal(theta, 0.0)
    rho = np.trace(pi_mat) + rbar * ((sd[None, :] / sd[:, None]) * theta).sum()
    gamma = ((F - S) ** 2).sum()                      # misspecification of the target
    delta = float(np.clip((pi_mat.sum() - rho) / gamma / T, 0.0, 1.0))
    return delta * F + (1 - delta) * S, delta, rbar


WINDOW, ANN = 252, 252
MONTH_ENDS = pd.to_datetime(monthly["MonthEnd"])      # Period('YYYY-MM') -> last NYSE session


def cov_at(month, window=WINDOW):
    """Annualized LW-CC covariance known at the close of decision month t (no data after month-end t)."""
    end = MONTH_ENDS.loc[pd.Period(month, "M")]
    hist = daily.loc[:end]
    hist = hist.loc[hist["regular"].astype(bool), TICKERS].tail(window)   # irregular returns excluded
    if len(hist) < window or hist.isna().any().any() or hist.index[-1] != end:
        raise ValueError(f"{month}: incomplete covariance window ({len(hist)} obs, last {hist.index[-1].date()})")
    Sigma, delta, rbar = ledoit_wolf_cc(hist.values)
    S = hist.cov(ddof=0).values
    return (pd.DataFrame(Sigma * ANN, index=TICKERS, columns=TICKERS),
            pd.DataFrame(S * ANN, index=TICKERS, columns=TICKERS), delta, rbar)


# --- Latest decision month
m = views["Month"].max()
Sigma, S_sample, delta, rbar = cov_at(m)
vol = np.sqrt(np.diag(Sigma))
corr = Sigma / np.outer(vol, vol)
print(f"Decision month {m} (window ends {MONTH_ENDS[m].date()}): delta = {delta:.4f}, avg corr rbar = {rbar:.3f}")
print("Min eigenvalue (annual):", f"{np.linalg.eigvalsh(Sigma.values).min():.2e}")
print("Benchmark vol (annual):  ", f"{np.sqrt(BENCH @ Sigma @ BENCH):.2%}")
print("\nAnnualized vol:\n", pd.Series(vol, index=TICKERS).map("{:.1%}".format).to_string())
print("\nShrunk correlation:\n", corr.round(2).to_string())
print("\nMax |corr shrunk - corr sample|:",
      f"{np.abs(corr.values - (S_sample / np.outer(np.sqrt(np.diag(S_sample)), np.sqrt(np.diag(S_sample)))).values).max():.3f}")

# --- Every decision month of the evaluation period (first trade earns Jan 2016 -> decision Dec 2015)
dec_months = [p for p in MONTH_ENDS.index if pd.Period("2015-12", "M") <= p <= m]
COV = {p: cov_at(p) for p in dec_months}               # cached for the next cells
d = pd.Series({p: v[2] for p, v in COV.items()})
r = pd.Series({p: v[3] for p, v in COV.items()})
print(f"\n{len(COV)} decision months {dec_months[0]} -> {dec_months[-1]}")
print(pd.DataFrame({"delta": d.describe(), "rbar": r.describe()}).T[["min", "25%", "50%", "75%", "max"]].round(3))

Decision month 2026-10 (window ends 2026-10-30): delta = 0.0895, avg corr rbar = 0.447
Min eigenvalue (annual): 3.32e-04
Benchmark vol (annual):   9.81%

Annualized vol:
 SPY    12.8%
EFA    15.8%
EEM    25.5%
IEF     5.0%
LQD     5.6%
TLT     9.6%
VNQ    13.7%
GLD    28.5%

Shrunk correlation:
       SPY   EFA   EEM   IEF   LQD   TLT   VNQ   GLD
SPY  1.00  0.76  0.74  0.31  0.46  0.29  0.27  0.39
EFA  0.76  1.00  0.76  0.45  0.53  0.40  0.41  0.49
EEM  0.74  0.76  1.00  0.31  0.40  0.26  0.10  0.47
IEF  0.31  0.45  0.31  1.00  0.86  0.87  0.46  0.24
LQD  0.46  0.53  0.40  0.86  1.00  0.86  0.44  0.24
TLT  0.29  0.40  0.26  0.87  0.86  1.00  0.41  0.15
VNQ  0.27  0.41  0.10  0.46  0.44  0.41  1.00  0.16
GLD  0.39  0.49  0.47  0.24  0.24  0.15  0.16  1.00

Max |corr shrunk - corr sample|: 0.041

131 decision months 2015-12 -> 2026-10
         min    25%    50%    75%    max
delta  0.021  0.047  0.073  0.105  0.236
rbar   0.122  0.219  0.284  0.405  0.504


In [3]:
# Black-Litterman: implied (equilibrium) excess returns and posterior given views
def implied_excess_returns(Sigma, b, sr):
    """LN4: pi = mu_tilde - r = phi * Sigma b, with phi = SR(b) / sigma(b)."""
    Sigma = np.asarray(Sigma); b = np.asarray(b)
    sigma_b = np.sqrt(b @ Sigma @ b)
    phi = sr / sigma_b
    return phi * Sigma @ b, phi, sigma_b


def bl_posterior(mu_prior, Sigma, P, Q, conf, tau=0.05):
    """LN4 conditional mean with Gamma = tau*Sigma and Idzorek/Walters Omega_k = (1-c_k)/c_k * tau * P_k Sigma P_k'.
    conf: scalar or (k,) confidences in (0, 1]. Returns mu_bar (same units as mu_prior)."""
    Sigma = np.asarray(Sigma); P = np.atleast_2d(np.asarray(P, float)); Q = np.asarray(Q, float)
    c = np.broadcast_to(np.asarray(conf, float), Q.shape)
    if np.any(c <= 0) or np.any(c > 1):
        raise ValueError("confidence must be in (0, 1]")
    Gamma = tau * Sigma
    Omega = np.diag((1 - c) / c * np.diag(P @ Gamma @ P.T))
    return mu_prior + Gamma @ P.T @ np.linalg.solve(P @ Gamma @ P.T + Omega, Q - P @ mu_prior)


# --- Reference test: LN4 Example 4 (prior only) — expected mu_tilde = 5.47%, 6.68%, 8.70%, 9.06%, phi = 1.63
s_ex = np.array([0.15, 0.20, 0.25, 0.30])
C_ex = np.array([[1, .1, .4, .5], [.1, 1, .7, .4], [.4, .7, 1, .8], [.5, .4, .8, 1]])
pi_ex, phi_ex, sb_ex = implied_excess_returns(C_ex * np.outer(s_ex, s_ex), [.4, .3, .2, .1], sr=0.25)
assert np.allclose(0.03 + pi_ex, [0.0547, 0.0668, 0.0870, 0.0906], atol=5e-5) and abs(phi_ex - 1.63) < 5e-3
print(f"LN4 Example 4 reproduced: sigma(x0) = {sb_ex:.2%}, phi = {phi_ex:.2f}, mu_tilde = {np.round((0.03 + pi_ex) * 100, 2)}")


def views_for(month):
    v = views.loc[views["Month"] == pd.Period(month, "M")]
    return v[TICKERS].values, v["Q"].values, v["View"].tolist()


# --- Latest decision month, placeholder SR and confidence
SR, CONF = 0.40, 0.50
P, Q, names = views_for(m)
pi, phi, sigma_b = implied_excess_returns(Sigma, BENCH, SR)
mu_bar = bl_posterior(pi, Sigma, P, Q, CONF)
alpha = mu_bar - pi

print(f"\n{m}: sigma_b = {sigma_b:.2%}, SR = {SR}, phi = {phi:.2f}, confidence = {CONF}")
print(pd.DataFrame({"Q (view)": Q, "P·pi (prior spread)": P @ pi, "Q - P·pi": Q - P @ pi,
                    "P·mu_bar (posterior)": P @ mu_bar}, index=names).map("{:+.2%}".format).to_string())
print("\nPer asset (annual excess returns):")
print(pd.DataFrame({"pi (prior)": pi, "mu_bar": mu_bar, "alpha = mu_bar - pi": alpha}, index=TICKERS)
      .map("{:+.2%}".format).to_string())

# --- Sanity checks
a_tau = bl_posterior(pi, Sigma, P, Q, CONF, tau=0.5) - pi
assert np.allclose(a_tau, alpha), "tau should cancel with Idzorek/Walters Omega"
assert np.allclose(bl_posterior(pi, Sigma, P, P @ pi, CONF), pi), "views equal to prior must give alpha = 0"
assert np.allclose(P @ bl_posterior(pi, Sigma, P, Q, 1.0), Q), "100% confidence must give P·mu_bar = Q"
print("\nChecks passed: tau cancels | Q = P·pi -> alpha = 0 | conf = 100% -> P·mu_bar = Q")

# --- How much the SR choice moves the prior view spreads (decision 3)
print("\nPrior spread P·pi by benchmark SR (compare with Q):")
print(pd.DataFrame({f"SR={s}": P @ implied_excess_returns(Sigma, BENCH, s)[0] for s in [0.2, 0.3, 0.4, 0.5]}
                   | {"Q": Q}, index=names).map("{:+.2%}".format).to_string())

LN4 Example 4 reproduced: sigma(x0) = 15.35%, phi = 1.63, mu_tilde = [5.47 6.68 8.7  9.06]

2026-10: sigma_b = 9.81%, SR = 0.4, phi = 4.08, confidence = 0.5
                                 Q (view) P·pi (prior spread) Q - P·pi P·mu_bar (posterior)
Momentum Equities: EEM > SPY       +1.62%              +3.92%   -2.30%               +2.01%
Momentum Bonds: LQD > TLT          +0.97%              -0.53%   +1.50%               +0.13%
Momentum Alternatives: VNQ > GLD   +0.14%              -4.54%   +4.68%               -1.81%
Trend: equities vs bonds           +1.44%              +3.90%   -2.47%               +2.70%

Per asset (annual excess returns):
    pi (prior)  mu_bar alpha = mu_bar - pi
SPY     +4.51%  +3.05%              -1.45%
EFA     +5.68%  +3.69%              -1.99%
EEM     +8.43%  +5.06%              -3.37%
IEF     +1.16%  +0.67%              -0.48%
LQD     +1.51%  +1.00%              -0.51%
TLT     +2.04%  +0.87%              -1.17%
VNQ     +2.47%  +2.06%              -0.40%
GLD

In [4]:
# Diagnostic over all decision months: does the BL tilt go in the direction of the signal?
# textbook : Q is the expected spread itself (LN4)                  -> tilt driven by Q - P·pi
# relative : Q is the spread ON TOP of equilibrium, Q_eff = P·pi + Q -> tilt driven by Q
rows = []
for p in dec_months:
    Sigma_p = COV[p][0]
    P_p, Q_p, names_p = views_for(p)
    for sr in [0.2, 0.4]:
        pi_p = implied_excess_returns(Sigma_p, BENCH, sr)[0]
        for mode, Q_eff in [("textbook", Q_p), ("relative", P_p @ pi_p + Q_p)]:
            shift = P_p @ (bl_posterior(pi_p, Sigma_p, P_p, Q_eff, CONF) - pi_p)   # posterior tilt per view
            for n, q, s in zip(names_p, Q_p, shift):
                rows.append({"Month": p, "SR": sr, "mode": mode, "view": n.split(":")[0], "Q": q, "shift": s})

diag = pd.DataFrame(rows)
diag = diag[diag["Q"].abs() > 1e-12]                       # a zero signal has no direction
diag["against"] = np.sign(diag["shift"]) != np.sign(diag["Q"])
summary = diag.groupby(["mode", "SR", "view"]).agg(
    against_signal=("against", "mean"),
    median_Q=("Q", lambda x: x.abs().median()),
    median_shift=("shift", lambda x: x.abs().median()),
    corr_Q_shift=("shift", lambda s: np.corrcoef(s, diag.loc[s.index, "Q"])[0, 1]),
)
print("Share of months where the posterior tilt goes AGAINST the signal, and |Q| vs |tilt| (annual):")
print(summary.to_string(formatters={"against_signal": "{:.0%}".format, "median_Q": "{:.2%}".format,
                                    "median_shift": "{:.2%}".format, "corr_Q_shift": "{:+.2f}".format}))

Share of months where the posterior tilt goes AGAINST the signal, and |Q| vs |tilt| (annual):
                                   against_signal median_Q median_shift corr_Q_shift
mode     SR  view                                                                   
relative 0.2 Momentum Alternatives            11%    2.00%        0.96%        +0.84
             Momentum Bonds                    2%    1.13%        0.59%        +0.87
             Momentum Equities                 0%    2.12%        1.05%        +0.97
             Trend                            11%    2.45%        1.15%        +0.87
         0.4 Momentum Alternatives            11%    2.00%        0.96%        +0.84
             Momentum Bonds                    2%    1.13%        0.59%        +0.87
             Momentum Equities                 0%    2.12%        1.05%        +0.97
             Trend                            11%    2.45%        1.15%        +0.87
textbook 0.2 Momentum Alternatives            21%    2.0

In [5]:
# BL alpha (equilibrium-relative views) + optimizer (LN2 sigma-problem with mandate)
from scipy.optimize import minimize

TE_TARGET = 0.02


def bl_alpha(month, conf=0.5, sr=0.4):
    """alpha = mu_bar - pi with equilibrium-relative views Q_eff = P·pi + Q (sr then cancels out of alpha)."""
    Sigma_m = COV[pd.Period(month, "M")][0]
    P_m, Q_m, _ = views_for(month)
    pi_m = implied_excess_returns(Sigma_m, BENCH, sr)[0]
    mu_bar = bl_posterior(pi_m, Sigma_m, P_m, P_m @ pi_m + Q_m, conf)
    return pd.Series(mu_bar - pi_m, index=TICKERS), Sigma_m


def linear_rules(rules):
    """Mandate as G x <= h.  rules: 'full' (mandate), 'long_only' (x >= 0 only), 'none' (sum = 1 only)."""
    n, I = len(TICKERS), np.eye(len(TICKERS))
    ind = lambda names: np.array([t in names for t in TICKERS], float)
    G, h = [], []
    if rules in ("full", "long_only"):
        G += list(-I); h += [0.0] * n                            # x_i >= 0
    if rules == "full":
        G += list(I); h += [0.40] * n                            # x_i <= 40%
        G += [-ind(BONDS), ind(BONDS)]; h += [-0.25, 0.45]       # 25% <= bonds <= 45%
        G += [ind(["EEM"]) - ind(EQUITY) / 3]; h += [0.0]        # EEM <= 1/3 of equities
    return (np.array(G), np.array(h)) if G else (np.zeros((0, n)), np.zeros(0))


def check_rules(x, rules="full", tol=1e-6):
    """Return the list of violated rules (empty list = valid portfolio)."""
    bad = []
    if abs(x.sum() - 1) > tol: bad.append("sum != 1")
    if rules in ("full", "long_only") and (x < -tol).any(): bad.append("short position")
    if rules == "full":
        if (x > 0.40 + tol).any(): bad.append("weight > 40%")
        if not (0.25 - tol <= x[BONDS].sum() <= 0.45 + tol): bad.append("bonds outside 25-45%")
        if x["EEM"] > x[EQUITY].sum() / 3 + tol: bad.append("EEM > 1/3 equity")
    return bad


def optimize_active(alpha, Sigma, rules="full", te=TE_TARGET, b=BENCH):
    """max alpha'(x - b)  s.t.  1'x = 1,  (x-b)' Sigma (x-b) <= te^2,  G x <= h.  Solved in active weights a = x - b."""
    a_vec, S, b_vec = np.asarray(alpha, float), np.asarray(Sigma, float), np.asarray(b, float)
    norm = np.linalg.norm(a_vec)
    if norm < 1e-12:                                             # no conviction -> hold the benchmark
        return pd.Series(b_vec, index=TICKERS), True
    d = a_vec / norm                                             # only the direction of alpha matters
    G, h = linear_rules(rules)
    cons = [{"type": "eq", "fun": lambda a: a.sum(), "jac": lambda a: np.ones_like(a)},
            {"type": "ineq", "fun": lambda a: 1 - a @ S @ a / te**2, "jac": lambda a: -2 * S @ a / te**2}]
    if len(h):
        cons.append({"type": "ineq", "fun": lambda a: h - G @ (b_vec + a), "jac": lambda a: -G})
    res = minimize(lambda a: -d @ a, np.zeros_like(b_vec), jac=lambda a: -d,
                   method="SLSQP", constraints=cons, options={"ftol": 1e-12, "maxiter": 500})
    x = b_vec + res.x
    x[np.abs(x) < 1e-9] = 0.0                                    # remove solver noise (e.g. -1e-12)
    x = pd.Series(x, index=TICKERS)
    # SLSQP can flag 'positive directional derivative' AT the optimum when many constraints bind:
    # accept the solution if it is feasible and on the TE budget
    feasible = not check_rules(x, rules) and np.sqrt(res.x @ S @ res.x) <= te * (1 + 1e-6)
    return x, bool(res.success) or feasible


# --- Test 1: unconstrained case vs closed form  a* = te * Sigma^-1(alpha - lambda 1) / ||.||_Sigma
alpha_m, Sigma_m = bl_alpha(m, CONF)
Si = np.linalg.inv(Sigma_m.values); one = np.ones(len(TICKERS))
lam = (one @ Si @ alpha_m) / (one @ Si @ one)
a_cf = Si @ (alpha_m.values - lam * one); a_cf *= TE_TARGET / np.sqrt(a_cf @ Sigma_m.values @ a_cf)
x_none, ok = optimize_active(alpha_m, Sigma_m, rules="none")
print(f"Closed-form check (rules='none'): max |w_slsqp - w_closed| = {np.abs(x_none.values - BENCH.values - a_cf).max():.1e}, success={ok}")

# --- Latest month: three portfolios
rows = {}
for rules in ["none", "long_only", "full"]:
    x, ok = optimize_active(alpha_m, Sigma_m, rules)
    a = x - BENCH
    rows[rules] = {**x.to_dict(), "TE": np.sqrt(a @ Sigma_m @ a), "E[active]": alpha_m @ a,
                   "violations": ", ".join(check_rules(x, rules)) or "-", "success": ok}
out = pd.DataFrame(rows).T
print(f"\n{m}, confidence {CONF}: alpha (annual) = " + ", ".join(f"{t} {v:+.2%}" for t, v in alpha_m.items()))
print(pd.concat([BENCH.rename("bench").to_frame().T, out]).to_string(
    formatters={**{t: "{:.1%}".format for t in TICKERS}, "TE": "{:.2%}".format, "E[active]": "{:+.2%}".format}))

# --- Every decision month, full mandate and long-only
stats, W_full = [], {}
for p in dec_months:
    al, Sg = bl_alpha(p, CONF)
    for rules in ["long_only", "full"]:
        x, ok = optimize_active(al, Sg, rules)
        a = (x - BENCH).values
        if rules == "full": W_full[p] = x
        stats.append({"Month": p, "rules": rules, "success": ok, "violations": len(check_rules(x, rules)),
                      "TE": np.sqrt(a @ Sg.values @ a), "E_active": al.values @ a, "max_w": x.max()})
stats = pd.DataFrame(stats)
print("\nAll decision months:")
print(stats.groupby("rules").agg(months=("Month", "size"), solver_ok=("success", "mean"),
                                 months_with_violation=("violations", lambda v: int((v > 0).sum())),
                                 TE_min=("TE", "min"), TE_median=("TE", "median"), TE_max=("TE", "max"),
                                 E_active_median=("E_active", "median")).round(4).to_string())

W_full = pd.DataFrame(W_full).T
turnover = W_full.diff().abs().sum(axis=1) / 2                  # one-way, ignores drift (C's backtest does it properly)
print(f"\nFull mandate: one-way turnover/month median {turnover.median():.1%}, mean {turnover.mean():.1%}"
      f" | active share sum|x-b|/2 median {((W_full - BENCH).abs().sum(axis=1) / 2).median():.1%}")
print("Weight range across months:")
print(W_full.agg(["min", "mean", "max"]).map("{:.1%}".format).to_string())

Closed-form check (rules='none'): max |w_slsqp - w_closed| = 5.3e-09, success=True

2026-10, confidence 0.5: alpha (annual) = SPY +0.48%, EFA +0.49%, EEM +1.36%, IEF -0.23%, LQD -0.15%, TLT -0.65%, VNQ -0.20%, GLD +0.31%
            SPY   EFA   EEM   IEF   LQD    TLT  VNQ  GLD    TE E[active] violations success
bench     30.0% 12.0%  8.0% 15.0% 10.0%  10.0% 8.0% 7.0%   NaN       NaN        NaN     NaN
none      29.0% 13.2% 12.9% 12.8% 36.8% -19.7% 9.4% 5.6% 2.00%    +0.22%          -    True
long_only 34.4% 11.8% 14.6%  3.8% 21.0%   0.0% 8.8% 5.6% 2.00%    +0.18%          -    True
full      34.3% 11.8% 14.6%  3.7% 21.3%   0.0% 8.7% 5.6% 2.00%    +0.18%          -    True

All decision months:
           months  solver_ok  months_with_violation  TE_min  TE_median  TE_max  E_active_median
rules                                                                                          
full          131        1.0                      0    0.02       0.02    0.02           0.0026
long_only

In [6]:
# Turnover cap (LN3 definition: sum_i |x_i - x_prev_i| <= cap, x_prev = drifted pre-trade weights)
MAX_TURNOVER = 0.20


def optimize_active_tc(alpha, Sigma, rules="full", te=TE_TARGET, b=BENCH, x_prev=None, max_turnover=None):
    """Same sigma-problem, plus LN3 turnover cap sum|x - x_prev| <= max_turnover (Scherer: x = x_prev + u_plus - u_minus).
    Decision vector z = [a (active weights), u_plus, u_minus]."""
    if x_prev is None or max_turnover is None:
        return optimize_active(alpha, Sigma, rules, te, b)
    n = len(TICKERS)
    a_vec, S, b_vec, x0 = (np.asarray(v, float) for v in (alpha, Sigma, b, x_prev))
    norm = np.linalg.norm(a_vec)
    d = a_vec / norm if norm > 1e-12 else np.zeros(n)
    G, h = linear_rules(rules)
    A = lambda z: z[:n]
    cons = [{"type": "eq", "fun": lambda z: A(z).sum(), "jac": lambda z: np.r_[np.ones(n), np.zeros(2 * n)]},
            {"type": "ineq", "fun": lambda z: 1 - A(z) @ S @ A(z) / te**2,
             "jac": lambda z: np.r_[-2 * S @ A(z) / te**2, np.zeros(2 * n)]},
            {"type": "eq", "fun": lambda z: b_vec + A(z) - x0 - z[n:2*n] + z[2*n:],
             "jac": lambda z: np.hstack([np.eye(n), -np.eye(n), np.eye(n)])},
            {"type": "ineq", "fun": lambda z: max_turnover - z[n:].sum(),
             "jac": lambda z: np.r_[np.zeros(n), -np.ones(2 * n)]}]
    if len(h):
        cons.append({"type": "ineq", "fun": lambda z: h - G @ (b_vec + A(z)),
                     "jac": lambda z: np.hstack([-G, np.zeros((len(h), 2 * n))])})
    z0 = np.r_[x0 - b_vec, np.zeros(2 * n)]                       # start from 'no trade'
    bounds = [(None, None)] * n + [(0, None)] * (2 * n)
    res = minimize(lambda z: -d @ A(z), z0, jac=lambda z: np.r_[-d, np.zeros(2 * n)], bounds=bounds,
                   method="SLSQP", constraints=cons, options={"ftol": 1e-12, "maxiter": 1000})
    x = b_vec + res.x[:n]
    x[np.abs(x) < 1e-9] = 0.0
    x = pd.Series(x, index=TICKERS)
    te_x = np.sqrt((x - b_vec) @ S @ (x - b_vec))
    to = np.abs(x.values - x0).sum()
    feasible = not check_rules(x, rules) and te_x <= te * (1 + 1e-5) and to <= max_turnover + 1e-6
    return x, feasible


def propose_weights(alpha, Sigma, rules="full", x_prev=None, max_turnover=MAX_TURNOVER):
    """Priority: mandate and TE budget are hard; the turnover cap is relaxed (x1.25 steps) only if it makes
    the problem infeasible (e.g. Sigma jumps and the old portfolio is far outside the TE budget)."""
    x, ok = optimize_active_tc(alpha, Sigma, rules, x_prev=x_prev, max_turnover=max_turnover)
    cap = max_turnover
    while not ok and cap is not None and cap < 2.0:               # 2.0 = max possible two-way turnover
        cap = min(cap * 1.25, 2.0)
        x, ok = optimize_active_tc(alpha, Sigma, rules, x_prev=x_prev, max_turnover=cap)
    return x, ok, (cap if cap != max_turnover else None)


def run_path(cap, conf=CONF, rules="full"):
    """Sequential decisions; the fund starts at the benchmark; weights drift with next month's returns."""
    R = monthly[TICKERS]
    x_prev, out = BENCH.copy(), []
    for p in dec_months:
        al, Sg = bl_alpha(p, conf)
        x, ok, relaxed = propose_weights(al, Sg, rules, x_prev=x_prev, max_turnover=cap)
        a = (x - BENCH).values
        a_prev = (x_prev - BENCH).values
        out.append({"Month": p, "ok": ok, "relaxed_cap": relaxed, "TE": np.sqrt(a @ Sg.values @ a),
                    "TE_before_trade": np.sqrt(a_prev @ Sg.values @ a_prev),
                    "turnover": np.abs(x - x_prev).sum(), "active_share": np.abs(a).sum() / 2,
                    "E_active": al.values @ a, "violations": len(check_rules(x, rules)), **x.to_dict()})
        if p + 1 in R.index:                                      # drift to the next decision date
            grown = x * (1 + R.loc[p + 1])
            x_prev = grown / grown.sum()
    return pd.DataFrame(out).set_index("Month")


paths = {"no cap": run_path(None), "cap 20%": run_path(MAX_TURNOVER)}
summary = pd.DataFrame({k: {
    "solver ok": v["ok"].mean(),
    "rule violations": int((v["violations"] > 0).sum()),
    "months cap relaxed": int(v["relaxed_cap"].notna().sum()),
    "months turnover >= 20%": int((v["turnover"] > MAX_TURNOVER - 1e-4).sum()),
    "turnover mean (2-way)": v["turnover"].mean(),
    "turnover median (2-way)": v["turnover"].median(),
    "TE min": v["TE"].min(), "TE median": v["TE"].median(), "TE max": v["TE"].max(),
    "active share median": v["active_share"].median(),
    "E[active] median": v["E_active"].median(),
} for k, v in paths.items()})
print(summary.to_string(float_format=lambda f: f"{f:.4f}"))

capped = paths["cap 20%"]
print("\nMonths where the cap had to be relaxed or TE is below 1.99%:")
cols = ["relaxed_cap", "TE_before_trade", "TE", "turnover"]
print(capped.loc[capped["relaxed_cap"].notna() | (capped["TE"] < 0.0199), cols].round(4).to_string())

                         no cap  cap 20%
solver ok                1.0000   1.0000
rule violations          0.0000   0.0000
months cap relaxed       0.0000   1.0000
months turnover >= 20%  75.0000  83.0000
turnover mean (2-way)    0.2670   0.1742
turnover median (2-way)  0.2283   0.2000
TE min                   0.0200   0.0131
TE median                0.0200   0.0200
TE max                   0.0200   0.0200
active share median      0.2599   0.2377
E[active] median         0.0026   0.0026

Months where the cap had to be relaxed or TE is below 1.99%:
         relaxed_cap  TE_before_trade      TE  turnover
Month                                                  
2015-12          NaN           0.0000  0.0170      0.20
2018-04          NaN           0.0201  0.0197      0.20
2019-09          NaN           0.0218  0.0131      0.20
2019-12          NaN           0.0184  0.0176      0.20
2020-03         0.25           0.0491  0.0200      0.25


In [7]:
# The src/ modules must reproduce the notebook exactly (same path with the 20% turnover cap)
import importlib
import src.config, src.black_litterman, src.optimizer
for mod in (src.config, src.black_litterman, src.optimizer):
    importlib.reload(mod)                                        # pick up edits without restarting the kernel
from src.optimizer import load_inputs, propose_portfolio

inputs = load_inputs()
R = monthly[TICKERS]
x_prev, W_src, flags = BENCH.copy(), {}, {}
for p in dec_months:
    x, diag = propose_portfolio(p, inputs, x_prev=x_prev, rules="full", confidence=CONF)
    W_src[p] = x
    flags[p] = (diag["ok"], diag["relaxed_turnover_cap"], len(diag["violations"]))
    if p + 1 in R.index:
        grown = x * (1 + R.loc[p + 1])
        x_prev = grown / grown.sum()
W_src = pd.DataFrame(W_src).T

W_nb = paths["cap 20%"][TICKERS]
print("Max |w_src - w_notebook| over all months:", f"{(W_src - W_nb).abs().max().max():.1e}")
print("All ok:", all(f[0] for f in flags.values()), "| violations:", sum(f[2] for f in flags.values()),
      "| relaxed months:", [str(p) for p, f in flags.items() if f[1] is not None])

# What the app / backtest gets back for one month (with a manual view)
x, diag = propose_portfolio(m, inputs, rules="full", confidence=0.5,
                            manual_view={"long": "GLD", "short": "TLT", "q": 0.03, "confidence": 0.7})
print(f"\n{m} with manual view GLD > TLT (+3% over equilibrium, 70%): TE {diag['te_ex_ante']:.2%}, "
      f"E[active] {diag['expected_active_return']:+.2%}, delta {diag['shrinkage_delta']:.3f}")
print(pd.DataFrame({"bench": BENCH, "proposed": x}).T.map("{:.1%}".format).to_string())
print(diag["views"].map("{:+.2%}".format).to_string())

Max |w_src - w_notebook| over all months: 1.5e-08
All ok: True | violations: 0 | relaxed months: ['2020-03']

2026-10 with manual view GLD > TLT (+3% over equilibrium, 70%): TE 2.00%, E[active] +0.20%, delta 0.089
            SPY    EFA    EEM    IEF    LQD    TLT   VNQ   GLD
bench     30.0%  12.0%   8.0%  15.0%  10.0%  10.0%  8.0%  7.0%
proposed  31.2%  10.9%  13.7%   4.0%  21.0%   0.0%  9.4%  9.8%
                                 Q_over_equilibrium confidence prior_spread posterior_spread
Momentum Equities: EEM > SPY                 +1.62%    +50.00%       +3.92%           +4.87%
Momentum Bonds: LQD > TLT                    +0.97%    +50.00%       -0.53%           +0.00%
Momentum Alternatives: VNQ > GLD             +0.14%    +50.00%       -4.54%           -5.96%
Trend: equities vs bonds                     +1.44%    +50.00%       +3.90%           +4.98%
Manual: GLD > TLT                            +3.00%    +70.00%       +4.97%           +7.05%
